<a href="https://colab.research.google.com/github/LuizFellipiFreire25/Projeto-ECAA08/blob/main/09%20-%20Motor%20de%20Inferencia%20Forward%20e%20Backward%20Chaining.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Aula 09 - Notebook: Motores de Inferência Forward e Backward Chaining do AGV

Neste notebook implementamos os algoritmos completos de **Forward Chaining** (orientado a dados de telemetria) e **Backward Chaining** (orientado a diagnóstico de hipóteses) para o SCADA-Core do AGV.

---

### Célula 1 (Texto / Markdown)
```markdown
## 1. Módulo Auxiliar e Estruturas Globais

Implementação da função de formatação de tabelas em ASCII puro, dataclass de `RegraProducao`, container de `BaseConhecimento` e o motor híbrido de inferência `MotorInferencia`.

In [1]:
from dataclasses import dataclass
from typing import Set, Tuple, List, Dict, Optional, Any

def formatar_tabela(dados: List[Dict[str, Any]]) -> str:
    """Formata lista de dicionários em tabela ASCII pura."""
    if not dados:
        return "Tabela Vazia"
    colunas = list(dados[0].keys())
    larguras = {c: len(str(c)) for c in colunas}
    for row in dados:
        for c in colunas:
            larguras[c] = max(larguras[c], len(str(row.get(c, ""))))
    header = " | ".join(f"{c:<{larguras[c]}}" for c in colunas)
    divisor = "-+-".join("-" * larguras[c] for c in colunas)
    linhas = [header, divisor]
    for row in dados:
        linhas.append(" | ".join(f"{str(row.get(c, '')):<{larguras[c]}}" for c in colunas))
    return "\n".join(linhas)

@dataclass
class RegraProducao:
    id_regra: str
    antecedentes: Set[str]
    consequente: str
    descricao_diagnostico: str
    prioridade: int = 1

class BaseConhecimento:
    def __init__(self):
        self.regras: List[RegraProducao] = []

    def adicionar_regra(self, id_r: str, antecedentes: List[str], consequente: str, desc: str, prioridade: int = 1):
        self.regras.append(RegraProducao(id_r, set(antecedentes), consequente, desc, prioridade))

class MotorInferencia:
    def __init__(self, base_conhecimento: BaseConhecimento):
        self.bc = base_conhecimento

    def forward_chaining(self, fatos_iniciais: Set[str]) -> Tuple[Set[str], List[Dict[str, Any]]]:
        """Encadeamento para Frente: Reativo / Data-Driven"""
        fatos_conhecidos = set(fatos_iniciais)
        historico_disparos = []
        novos_fatos = True
        passo = 1

        while novos_fatos:
            novos_fatos = False
            regras_candidatas = sorted(self.bc.regras, key=lambda r: r.prioridade, reverse=True)
            for regra in regras_candidatas:
                if regra.antecedentes.issubset(fatos_conhecidos) and regra.consequente not in fatos_conhecidos:
                    fatos_conhecidos.add(regra.consequente)
                    historico_disparos.append({
                        "Passo": passo,
                        "Regra": regra.id_regra,
                        "Fato Inferido": regra.consequente,
                        "Diagnóstico": regra.descricao_diagnostico
                    })
                    passo += 1
                    novos_fatos = True
                    break
        return fatos_conhecidos, historico_disparos

    def backward_chaining(self, meta: str, fatos_conhecidos: Set[str], historico: Optional[List[Dict[str, Any]]] = None) -> Tuple[bool, List[Dict[str, Any]]]:
        """Encadeamento para Trás: Diagnóstico / Goal-Driven"""
        if historico is None:
            historico = []

        if meta in fatos_conhecidos:
            historico.append({"Meta/Submeta": meta, "Status": "CONFIRMADO (Fato de Campo)", "Regra": "N/A"})
            return True, historico

        regras_candidatas = [r for r in self.bc.regras if r.consequente == meta]
        regras_candidatas.sort(key=lambda r: r.prioridade, reverse=True)

        for regra in regras_candidatas:
            todas_submetas_ok = True
            for antecedente in regra.antecedentes:
                sub_ok, _ = self.backward_chaining(antecedente, fatos_conhecidos, historico)
                if not sub_ok:
                    todas_submetas_ok = False
                    break

            if todas_submetas_ok:
                fatos_conhecidos.add(meta)
                historico.append({
                    "Meta/Submeta": meta,
                    "Status": "PROVADO VIA REGRA",
                    "Regra": regra.id_regra
                })
                return True, historico

        historico.append({"Meta/Submeta": meta, "Status": "FALHA DE PROVA", "Regra": "N/A"})
        return False, historico

print("MotorInferencia Híbrido (Forward/Backward) carregado com sucesso.")

MotorInferencia Híbrido (Forward/Backward) carregado com sucesso.


## 2. Teste 1: Forward Chaining (Atuação Reativa de Emergência)

Simulação de eventos de campo em tempo real onde a entrada dos fatos `lidar_zona_vermelha`, `velocidade_gt_zero` e `falha_comunic_wifi` dispara uma reação em cadeia até o isolamento total da tração do AGV (`trip_isolamento_total`).

In [2]:
# Instanciando a Base de Conhecimento com regras de segurança do AGV
bc_agv = BaseConhecimento()
bc_agv.adicionar_regra("R-01", ["lidar_zona_vermelha", "velocidade_gt_zero"], "risco_colisao_iminente", "Detecção de Obstáculo em Rota de Impacto", 10)
bc_agv.adicionar_regra("R-02", ["risco_colisao_iminente", "falha_comunic_wifi"], "trip_isolamento_total", "AGV Cego e Incomunicável - Interrupção Total de Tração", 10)
bc_agv.adicionar_regra("R-03", ["motor_pwm_high", "encoder_rpm_zero"], "rotor_bloqueado", "Desarme do Motor M301 por Sobrecorrente", 8)
bc_agv.adicionar_regra("R-04", ["bms_temp_high", "corrente_carga_high"], "risco_fuga_termica", "Corte de Carga por Sobreaquecimento do Pack Lítio", 9)

motor_agv = MotorInferencia(bc_agv)

# Fatos iniciais de telemetria recebidos no SCADA
telemetria_campo = {"lidar_zona_vermelha", "velocidade_gt_zero", "falha_comunic_wifi"}

fatos_finais, trilha_forward = motor_agv.forward_chaining(telemetria_campo)

print("Trilha de Diagnóstico Forward Chaining (Reação em Cadeia no AGV):")
print(formatar_tabela(trilha_forward))

# Testes formais automatizados
assert "risco_colisao_iminente" in fatos_finais
assert "trip_isolamento_total" in fatos_finais
print("\n[VALIDADO] O estado 'trip_isolamento_total' foi inferido com sucesso via Forward Chaining.")

Trilha de Diagnóstico Forward Chaining (Reação em Cadeia no AGV):
Passo | Regra | Fato Inferido          | Diagnóstico                                           
------+-------+------------------------+-------------------------------------------------------
1     | R-01  | risco_colisao_iminente | Detecção de Obstáculo em Rota de Impacto              
2     | R-02  | trip_isolamento_total  | AGV Cego e Incomunicável - Interrupção Total de Tração

[VALIDADO] O estado 'trip_isolamento_total' foi inferido com sucesso via Forward Chaining.


## 3. Teste 2: Backward Chaining (Investigação de Causa-Raiz Pós-Evento)

Dada uma pergunta investigativa realizada pelo operador do SCADA (*"O AGV está sob Trip de Isolamento Total?"*), o algoritmo realiza a busca regressiva na árvore de inferência para verificar se os fatos de campo sustentam essa hipótese.

In [3]:
# Telemetria armazenada nos logs de bordo
fatos_gravados_bordo = {"lidar_zona_vermelha", "velocidade_gt_zero", "falha_comunic_wifi"}

hipotese_meta = "trip_isolamento_total"

# Executando a prova por Backward Chaining
hipotese_provada, trilha_backward = motor_agv.backward_chaining(hipotese_meta, fatos_gravados_bordo.copy())

print(f"Investigação de Causa-Raiz para Meta: '{hipotese_meta}'")
print(f"Hipótese Confirmada? -> {hipotese_provada}\n")
print("Rastreamento da Árvore de Inferência (Backward Chaining):")
print(formatar_tabela(trilha_backward))

assert hipotese_provada == True
print("\n[VALIDADO] A hipótese de causa-raiz foi provada com sucesso via Backward Chaining.")

Investigação de Causa-Raiz para Meta: 'trip_isolamento_total'
Hipótese Confirmada? -> True

Rastreamento da Árvore de Inferência (Backward Chaining):
Meta/Submeta           | Status                     | Regra
-----------------------+----------------------------+------
falha_comunic_wifi     | CONFIRMADO (Fato de Campo) | N/A  
lidar_zona_vermelha    | CONFIRMADO (Fato de Campo) | N/A  
velocidade_gt_zero     | CONFIRMADO (Fato de Campo) | N/A  
risco_colisao_iminente | PROVADO VIA REGRA          | R-01 
trip_isolamento_total  | PROVADO VIA REGRA          | R-02 

[VALIDADO] A hipótese de causa-raiz foi provada com sucesso via Backward Chaining.
